# Figure 3 — The flow operator from a time course alone (S10, S12, S13, S14, S14b, S15)

Sources (column meanings in each RESULT_*.md):
* `benchmarks/synthetic_multimodal/s14/test.csv` — held-out test seeds; `configuration_id`
  `reference` (a-priori defaults) vs `selected` (tuned one factor at a time on dev seeds);
  `auc_lam1` = AUC of λ₁ saddle vs committed; transition_* = post-hoc binned-λ₁ profile.
* `benchmarks/synthetic_multimodal/s14/dev_auc_by_k.csv`, `gates.json`.
* `benchmarks/synthetic_multimodal/s10/operators.csv` — operator comparison on calibrated v3.
* `benchmarks/synthetic_multimodal/s12/step0.csv` — OT flow J AUC by embedding.
* `benchmarks/synthetic_multimodal/s13/auc.csv` — whitening variants.
* `benchmarks/synthetic_multimodal/s15/jref_diagnosis.csv` (final S15; `S15_DIR`).
* `benchmarks/synthetic_multimodal/s14c/g2_transition_eligibility.csv` (S14b reserve s5–s16 reused + S14c s17–s40).

In [1]:
from _style import *
import json as _json

# S15 folder: merged final S15 (benchmarks/synthetic_multimodal/s15)
S15 = S15_DIR
prov = Provenance("Fig3", notebook="notebooks/fig3_flow_operator.ipynb")
B = "benchmarks/synthetic_multimodal"
test = pd.read_csv(prov.use(f"{B}/s14/test.csv", "a,e", "S14"))
dev = pd.read_csv(prov.use(f"{B}/s14/dev_auc_by_k.csv", "a", "S14"))
gates14 = _json.loads(prov.use(f"{B}/s14/gates.json", "a,e", "S14").read_text())
ops = pd.read_csv(prov.use(f"{B}/s10/operators.csv", "b", "S10"))
s12 = pd.read_csv(prov.use(f"{B}/s12/step0.csv", "c", "S12"))
s13 = pd.read_csv(prov.use(f"{B}/s13/auc.csv", "c", "S13"))
jref = pd.read_csv(prov.use(S15 / "jref_diagnosis.csv", "d", "S15", note="final S15"))
_s15md = prov.use(S15 / "RESULT_S15.md", "d", "S15", note="status line (complete)").read_text()
assert "Complete:" in _s15md.split("## Status", 1)[1].split("##", 1)[0], "S15 not complete"
_s14c = prov.use(f"{B}/s14c/RESULT_S14c.md", "e", "S14c", note="outcome line (construction failure)").read_text()
assert "construction failure" in _s14c.splitlines()[0] and "2 of the required 4" in _s14c
elig = pd.read_csv(prov.use(f"{B}/s14c/g2_transition_eligibility.csv", "e", "S14b/S14c", note="S14b reserve s5-s16 reused + S14c extended reserve s17-s40"))
_e14b = pd.read_csv(prov.use(f"{B}/s14b/g2_transition_eligibility.csv", "e", "S14b", note="cross-check of the reused S14b rows"))
assert (_e14b.set_index("seed").bootstrap_stability.fillna(-1) == elig.set_index("seed").loc[_e14b.seed].bootstrap_stability.fillna(-1)).all()
syn = test[test.kind == "synthetic"].copy()
V = prov.value

fig = plt.figure(figsize=(DOUBLE_COL, 7.0))
gs0 = fig.add_gridspec(5, 6, height_ratios=[1, 0.22, 1.05, 0.62, 1.05], hspace=0.25, wspace=1.25)
class _Rows:  # map logical rows 0/1/2 to gridspec rows 0/2/4
    def __getitem__(self, k):
        r, c = k
        return gs0[2 * r, c]
gs = _Rows()
row3 = gs0[4, :].subgridspec(1, 3, width_ratios=[1.0, 0.62, 2.15], wspace=0.42)

## a — S14 held-out fork AUC vs k, reference vs selected

In [2]:
presets = ["v3_transition", "v3_default", "v3_batch", "v3_stress"]
axa = []
for i, pr in enumerate(presets):
    ax = fig.add_subplot(gs[0, i])
    sub = syn[syn.preset == pr]
    for ds, g in sub.groupby("dataset"):
        for cid, col, ls in [("reference", C_REF, "-"), ("selected", C_SELECTED, "--")]:
            h = g[g.configuration_id == cid].sort_values("k")
            ax.plot(h.k, h.auc_lam1, color=col, ls=ls, marker="o", ms=2.5, lw=0.9)
    ax.axhline(0.85, color=C_BASE, lw=0.6, ls=":")
    ax.set_xticks([10, 15, 20]); ax.set_ylim(0.35, 1.02)
    ax.set_title(f"{pr}\ntest seeds {', '.join(sorted(s.split('_s')[-1] for s in sub.dataset.unique()))}", fontsize=6)
    ax.set_xlabel("PCA k")
    if i == 0:
        ax.set_ylabel("fork AUC (λ₁ saddle vs committed)")
        panel_letter(ax, "a", x=-0.45)
    axa.append(ax)
ax = fig.add_subplot(gs[0, 4:])
ax.axis("off")
ax.legend(handles=[plt.Line2D([], [], color=C_REF, marker="o", ms=3, label="reference (a-priori defaults)"),
                   plt.Line2D([], [], color=C_SELECTED, ls="--", marker="o", ms=3, label="selected (tuned on 2 dev seeds)"),
                   plt.Line2D([], [], color=C_BASE, ls=":", label="G1 threshold 0.85")],
          loc="center left", fontsize=6)
ref = syn[syn.configuration_id == "reference"]; sel = syn[syn.configuration_id == "selected"]
tr = lambda d: d[d.preset == "v3_transition"]
ot = lambda d: d[d.preset.isin(["v3_default", "v3_batch", "v3_stress"])]
V("a_ref_transition_min", tr(ref).auc_lam1.min()); V("a_ref_transition_max", tr(ref).auc_lam1.max())
V("a_ref_other_min", ot(ref).auc_lam1.min()); V("a_ref_other_max", ot(ref).auc_lam1.max())
V("a_sel_transition_min", tr(sel).auc_lam1.min()); V("a_sel_transition_max", tr(sel).auc_lam1.max())
V("a_sel_other_min", ot(sel).auc_lam1.min()); V("a_sel_other_max", ot(sel).auc_lam1.max())
V("a_ref_all_min", ref.dropna(subset=["auc_lam1"]).auc_lam1.min()); V("a_ref_all_max", ref.auc_lam1.max())
V("a_n_test_datasets", int(ref.dropna(subset=["auc_lam1"]).dataset.nunique()))
V("a_gate_G1_min", gates14["gate_G1_auc_band"]["minimum_auc"])
V("a_dev_k15_transition", dev[(dev.k == 15) & (dev.preset == "v3_transition")].auc_lam1.tolist())
prov.panel("a", "S14 held-out fork AUC vs k (10/15/20), reference vs selected configuration, per test dataset (one line per seed)")

## b — S10 operator comparison on calibrated v3 (PCA)

In [3]:
axb = fig.add_subplot(gs[1, 0:2])
opl = [("J_flow_ot", "OT flow J", C_FLOW), ("J_flow_raw_shared", "labelling flow J\n(raw)", C_LABEL),
       ("J_flow_smoothed_shared", "labelling flow J\n(smoothed)", C_LABEL), ("J_flow_scvelo", "scVelo flow J", C_SCVELO),
       ("J_score", "score J", C_SCORE)]
mk = {"v3_default": "o", "v3_stress": "s", "v3_batch": "^", "v3_transition": "D"}
o = ops[(ops.method == "pca") & (ops.truth_class == "saddle")]
for j, (op, lab, col) in enumerate(opl):
    for pr, m_ in mk.items():
        v = o[(o.operator == op) & (o.preset == pr)].auc_saddle_vs_committed.values
        x = j + {"v3_default": -0.24, "v3_stress": -0.08, "v3_batch": 0.08, "v3_transition": 0.24}[pr]
        axb.scatter([x] * len(v), v, marker=m_, s=10, color=col if pr == "v3_default" else "white",
                    edgecolor=col, lw=0.7, zorder=3)
    d = o[(o.operator == op) & (o.preset == "v3_default")].auc_saddle_vs_committed
    V(f"b_{op}_v3default_min", d.min()); V(f"b_{op}_v3default_max", d.max())
lab_d = o[(o.operator.isin(["J_flow_raw_shared", "J_flow_smoothed_shared"])) & (o.preset == "v3_default")].auc_saddle_vs_committed
V("b_labelling_both_v3default_min", lab_d.min()); V("b_labelling_both_v3default_max", lab_d.max())
axb.axhline(0.5, color=C_BASE, lw=0.6, ls=":")
axb.set_xticks(range(len(opl))); axb.set_xticklabels([l for _, l, _ in opl], rotation=40, ha="right", fontsize=5.6)
axb.set_ylabel("fork AUC"); axb.set_ylim(0, 1.03)
axb.legend(handles=[plt.Line2D([], [], marker=m_, ls="", mfc="black" if p == "v3_default" else "white", mec="black", ms=3.5,
                               label=p.replace("v3_", "")) for p, m_ in mk.items()],
           fontsize=5, loc="lower left", ncol=2, handletextpad=0.2, columnspacing=0.5)
axb.set_title("S10: operator vs\nvelocity source (PCA-20)", fontsize=6.5)
panel_letter(axb, "b", x=-0.3)
prov.panel("b", "S10 fork AUC per operator (v3 presets, PCA-20, three seeds each; filled = v3_default)")

## c — representation dependence (S12 embeddings, S13 whitening)

In [4]:
axc = fig.add_subplot(gs[1, 2:4])
emb = [("pca10", "PCA-10"), ("pca20", "PCA-20"), ("pca50", "PCA-50"), ("scvi10", "scVI-10"), ("fa20", "FA-20"), ("fa20w", "FA-20\nwhitened cost")]
for j, (e, lab) in enumerate(emb):
    for pr, m_ in [("v3_default", "o"), ("v3_transition", "D")]:
        v = s12[(s12.embedding == e) & (s12.preset == pr)].auc_lam1.values
        axc.scatter(j + (-0.12 if pr == "v3_default" else 0.12) + np.zeros(len(v)), v, marker=m_, s=10,
                    color=C_FLOW if pr == "v3_default" else "white", edgecolor=C_FLOW, lw=0.7, zorder=3)
    v = s12[s12.embedding == e].auc_lam1
    V(f"c_s12_{e}_min", v.min()); V(f"c_s12_{e}_max", v.max())
axc.axhline(0.85, color=C_BASE, lw=0.6, ls=":")
axc.set_xticks(range(len(emb))); axc.set_xticklabels([l for _, l in emb], rotation=40, ha="right", fontsize=5.6)
axc.set_ylim(0, 1.03); axc.set_ylabel("OT flow J fork AUC")
axc.set_title("S12: embedding\n(default ●, transition ◇)", fontsize=6.5)
panel_letter(axc, "c", x=-0.3)
fa = s12[s12.embedding == "fa20"]

axc2 = fig.add_subplot(gs[1, 4:6])
var = [("raw_s12", "raw"), ("whitened_full", "whitened\nembedding"), ("raw_ot_whitened_drift", "whitened\ndrift fit only"),
       ("whitened_ot_raw_drift", "whitened\nOT cost only")]
embs = [("pca10", C_FLOW, "o"), ("pca20", "#3FA0D8", "s"), ("pca50", OKABE["skyblue"], "^"), ("fa20", C_BASE, "v")]
t13 = s13[s13.preset == "v3_transition"]
for j, (vv, lab) in enumerate(var):
    for i, (e, col, m_) in enumerate(embs):
        v = t13[(t13.variant == vv) & (t13.embedding == e)].auc_lam1.values
        axc2.scatter(j - 0.24 + 0.16 * i + np.zeros(len(v)), v, marker=m_, s=9, color=col, lw=0, zorder=3)
        V(f"c_s13_transition_{e}_{vv}_min", float(v.min())); V(f"c_s13_transition_{e}_{vv}_max", float(v.max()))
axc2.axhline(0.85, color=C_BASE, lw=0.6, ls=":")
axc2.set_xticks(range(len(var))); axc2.set_xticklabels([l for _, l in var], rotation=40, ha="right", fontsize=5.6)
axc2.set_ylim(0, 1.3); axc2.set_yticks(np.arange(0, 1.01, 0.2)); axc2.set_ylabel("fork AUC (transition s0–2)")
axc2.legend(handles=[plt.Line2D([], [], marker=m_, ls="", color=col, ms=3.5, label=e.upper().replace("PCA", "PCA-").replace("FA", "FA-"))
                     for e, col, m_ in embs], fontsize=6, loc="upper center", ncol=2, handletextpad=0.1, columnspacing=1.0, borderaxespad=0.1)
axc2.set_title("S13: whitening variants\n(v3_transition)", fontsize=6.5)
prov.panel("c", "S12 OT flow J fork AUC by embedding (6 datasets) and S13 whitening variants on v3_transition seeds 0-2")

## d — J_ref diagnosis (S15 step 0)

In [5]:
axd = fig.add_subplot(row3[0, 0])
jr = jref.sort_values(["dataset", "jref_neighbours"])
metrics = [("median_norm_J_over_norm_Jref", "‖J‖/‖J_ref‖"), ("median_per_cell_cosine", "per-cell cosine"),
           ("lambda1_spearman", "λ₁ Spearman"), ("median_relative_error_cstar_J", "rel. error\nafter c*")]
dsets = sorted(jr.dataset.unique())
cols_d = [OKABE["green"], C_FLOW, OKABE["skyblue"], OKABE["purple"]]
w = 0.18
for i, ds in enumerate(dsets):
    for nb, hatch in [(15, None), (30, "////")]:
        r = jr[(jr.dataset == ds) & (jr.jref_neighbours == nb)].iloc[0]
        for j, (mc, _) in enumerate(metrics):
            x = j + (i - 1.5) * w + (0.0 if nb == 15 else 0.0)
            if nb == 15:
                axd.scatter(x, r[mc], s=9, color=cols_d[i], zorder=3, lw=0)
            else:
                axd.scatter(x, r[mc], s=11, facecolor="white", edgecolor=cols_d[i], lw=0.7, zorder=3)
for mc, _ in metrics:
    V(f"d_{mc}_min", jr[mc].min()); V(f"d_{mc}_max", jr[mc].max())
V("d_fd_route_cosine_min", jr.basis_jref_fd_median_per_cell_cosine.min())
V("d_s14_jref_relfrob_min", syn.jref_relative_frobenius_median.min()); V("d_s14_jref_relfrob_max", syn.jref_relative_frobenius_median.max())
axd.axhline(0, color=C_BASE, lw=0.5); axd.axhline(1, color=C_BASE, lw=0.5, ls=":")
axd.set_xticks(range(len(metrics))); axd.set_xticklabels([l for _, l in metrics], fontsize=5.6, rotation=40, ha="right")
axd.set_ylim(-0.2, 1.1)
axd.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=c, ms=3, label=d.replace("v3_", "")) for d, c in zip(dsets, cols_d)]
           + [plt.Line2D([], [], marker="o", ls="", mfc="white", mec="black", ms=3, label="J_ref kNN 30")],
           fontsize=5, loc="center right", handletextpad=0.2)
axd.set_title("J_flow vs true J_ref\n(S15 step 0)", fontsize=6.5)
panel_letter(axd, "d", x=-0.3)
prov.panel("d", "J_ref diagnosis per test dataset (filled: J_ref with 15 neighbours, open: 30)")

## e — post-hoc transition peak-to-median R (S14, reference, k = 15) and S14b eligibility

In [6]:
axe = fig.add_subplot(row3[0, 1])
r15 = syn[(syn.configuration_id == "reference") & (syn.k == 15) & syn.preset.isin(["v3_transition", "v3_nofork"])]
for j, (pr, col) in enumerate([("v3_transition", C_FLOW), ("v3_nofork", C_BASE)]):
    h = r15[r15.preset == pr].sort_values("seed")
    axe.scatter([j] * len(h), h.transition_peak_to_median, color=col, s=14, zorder=3)
    for _, row in h.iterrows():
        txt = f"s{row.seed}" + (f" ({int(row.transition_abs_offset)})" if pr == "v3_transition" else "")
        axe.text(j + 0.08, row.transition_peak_to_median, txt, fontsize=5.2, va="center")
    V(f"e_R_{pr}_min", h.transition_peak_to_median.min()); V(f"e_R_{pr}_max", h.transition_peak_to_median.max())
    if pr == "v3_transition":
        V("e_transition_offsets", [int(x) for x in h.transition_abs_offset])
axe.set_xticks([0, 1]); axe.set_xticklabels(["transition", "no-fork"], rotation=30, ha="right", rotation_mode="anchor"); axe.set_xlim(-0.4, 1.9)
axe.set_ylabel("peak-to-median R")
axe.set_title("S14 post hoc\n(reference, k = 15)", fontsize=6.5)
panel_letter(axe, "e", x=-0.3)

axe2 = fig.add_subplot(row3[0, 2])
import json as _js
el = elig.sort_values("seed").reset_index(drop=True)
el["min_bin"] = el.bin_counts.map(lambda s_: min(_js.loads(s_)) if isinstance(s_, str) else np.nan)
T = lambda col: el[col].astype(str).eq("True")
done = el.generation_status.eq("completed")
eligible = T("eligible")
only_bins = done & ~eligible & ~T("all_bins_at_least_50_cells") & T("true_peak_bin_2_to_9") & T("bootstrap_stability_at_least_90pct") & T("strict_root_provenance_complete")
guard = el.generation_status.eq("guard_stop")
other_fail = ~done & ~guard
xs = np.arange(len(el))
for x_, r_ in zip(xs, el.itertuples()):
    if r_.generation_status != "completed":
        continue
    if str(r_.eligible) == "True":
        axe2.bar(x_, r_.bootstrap_stability, color=C_FLOW, width=0.75)
    elif only_bins[x_]:
        axe2.bar(x_, r_.bootstrap_stability, color="white", edgecolor=C_BASE, hatch="////", lw=0.6, width=0.75)
    else:
        axe2.bar(x_, r_.bootstrap_stability, color=C_BASE, width=0.75)
axe2.scatter(xs[guard], np.full(guard.sum(), 0.04), marker="x", color="#8B0000", s=12, lw=0.8, zorder=3)
axe2.scatter(xs[other_fail], np.full(other_fail.sum(), 0.04), marker="v", color="#8B0000", s=12, lw=0, zorder=3)
axe2.axhline(0.90, color="black", lw=0.7, ls="--")
split = xs[el.seed.values == 16][0] + 0.5
axe2.plot([split, split], [0, 1.05], color="#555555", lw=0.5, ls=":")
lab_seeds = {5, 10, 12, 16, 20, 25, 30, 35, 40}
axe2.set_xticks(xs, minor=True); axe2.tick_params(axis="x", which="minor", length=2)
axe2.set_xticks(xs[el.seed.isin(lab_seeds).values]); axe2.set_xticklabels([str(s_) for s_ in el.seed if s_ in lab_seeds], fontsize=6)
axe2.set_xlim(-0.7, len(el) - 0.3); axe2.set_ylim(0, 1.62); axe2.set_yticks(np.arange(0, 1.01, 0.2))
axe2.set_xlabel("v3_transition reserve seed (S14b: 5–16 | S14c: 17–40)"); axe2.set_ylabel("bootstrap stability\nof true peak bin")
axe2.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C_FLOW, label="eligible"),
                     plt.Rectangle((0, 0), 1, 1, color=C_BASE, label="not eligible"),
                     plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor=C_BASE, hatch="////", lw=0.6, label="fails only ≥ 50 cells/bin"),
                     plt.Line2D([], [], marker="x", ls="", color="#8B0000", ms=4, label="memory-guard stop"),
                     plt.Line2D([], [], marker="v", ls="", color="#8B0000", ms=4, label="other generation failure"),
                     plt.Line2D([], [], color="black", ls="--", lw=0.7, label="0.90 rule")],
           loc="upper left", ncol=2, fontsize=6, handlelength=1.3, handletextpad=0.3, columnspacing=0.8, borderaxespad=0.2,
           frameon=True, facecolor="white", edgecolor="none", framealpha=1).set_zorder(5)
axe2.set_title("S14b + S14c test construction (2 of 4 needed eligible)", fontsize=6.5)
V("e_s14b_n_assessed", int(done.sum())); V("e_s14b_n_eligible", int(eligible.sum()))
V("e_s14b_eligible_seeds", [int(s_) for s_ in el[eligible].seed])
V("e_s14b_n_reserve", int(len(el)))
V("e_n_guard_stop", int(guard.sum())); V("e_n_other_fail", int(other_fail.sum()))
V("e_other_fail_seeds", [int(s_) for s_ in el[other_fail].seed]); V("e_other_fail_status", sorted(set(el[other_fail].generation_status)))
V("e_only_bins_seeds", [int(s_) for s_ in el[only_bins].seed])
V("e_only_bins_min_bin", [int(m_) for m_ in el[only_bins].min_bin])
V("e_s14c_reserve_first", int(el.seed[el.seed > 16].min())); V("e_s14c_reserve_last", int(el.seed.max()))
V("e_s14c_n_assessed", int((done & (el.seed > 16)).sum())); V("e_s14c_n_eligible", int((eligible & (el.seed > 16)).sum()))
_g = _json.loads(prov.use(f"{B}/s14c/gates.json", "e", "S14c", note="gate values null (no model fitted)").read_text())
assert _g["eligible_seeds"] == V("e_s14b_eligible_seeds", [int(s_) for s_ in el[eligible].seed]) and _g["no_flow_fits"] and _g["overall"] is None
V("e_required_eligible", int(_g["required_eligible_seeds"]))
axe2.set_title(f"S14b + S14c test construction ({int(eligible.sum())} of {_g['required_eligible_seeds']} required seeds eligible)", fontsize=6.5)
prov.panel("e", "S14 post-hoc peak-to-median R (transition vs no-fork test seeds, reference k=15); S14b/S14c bootstrap stability per reserve seed s5-s40 with 0.90 rule, eligibility, seeds failing only the >=50-cells-per-bin rule, failed generations")

save(fig, "Fig3")

[save] main/Fig3: 162.1 x 159.0 mm, PNG 3830 x 3757 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig3.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig3.png'))

In [7]:
v = prov.values
R = lambda a, b: rng_str(v[a], v[b])
prov.legend(f"""
**Figure 3 | The flow operator from a time course alone ranks branching cells, untuned (S10, S12–S15, S14b).**
Flow J = Jacobian of MeasuredDrift fitted to entropic-OT barycentric velocities between sampling days (days 2/4/6) in PCA
of log-normalised counts. Fork AUC = AUC of the leading eigenvalue λ₁ for true saddle vs committed cells (truth from generator
state, used for scoring only).
**a**, S14 held-out test (pre-registered; test seeds never used for selection): {v['a_n_test_datasets']} test datasets
(v3_transition s2–s4, v3_default s2–s3, v3_batch s0–s2, v3_stress s0–s2) at k = 10, 15, 20. Reference (a-priori) configuration:
AUC {R('a_ref_transition_min', 'a_ref_transition_max')} on v3_transition and {R('a_ref_other_min', 'a_ref_other_max')} on the other presets.
Selected configuration (tuned one factor at a time on two development seeds): {R('a_sel_transition_min', 'a_sel_transition_max')} on
v3_transition (gate G1, ≥ 0.85 on every transition seed at k* = 15 and neighbours, failed; minimum {f2(v['a_gate_G1_min'])}).
**b**, S10 (pre-registered gate on labelling flow J; other operators report only): fork AUC per operator on v3 presets (PCA-20; 3 seeds
each; filled = v3_default). v3_default: OT flow J {R('b_J_flow_ot_v3default_min', 'b_J_flow_ot_v3default_max')}, labelling flow J
{R('b_labelling_both_v3default_min', 'b_labelling_both_v3default_max')} (raw and smoothed variants), scVelo flow J
{R('b_J_flow_scvelo_v3default_min', 'b_J_flow_scvelo_v3default_max')}, score J {R('b_J_score_v3default_min', 'b_J_score_v3default_max')}.
**c**, Left, S12 Step 0 (6 datasets: v3_default and v3_transition s0–s2): OT flow J fork AUC by embedding — PCA-10
{R('c_s12_pca10_min', 'c_s12_pca10_max')}, PCA-20 {R('c_s12_pca20_min', 'c_s12_pca20_max')}, PCA-50 {R('c_s12_pca50_min', 'c_s12_pca50_max')},
scVI-10 {R('c_s12_scvi10_min', 'c_s12_scvi10_max')}, FA-20 {R('c_s12_fa20_min', 'c_s12_fa20_max')}, FA-20 with noise-whitened cost
{R('c_s12_fa20w_min', 'c_s12_fa20w_max')}. Right, S13 (pre-registered; gate failed) on v3_transition s0–s2: whitening the embedding
(PCA-20 {R('c_s13_transition_pca20_whitened_full_min', 'c_s13_transition_pca20_whitened_full_max')}), whitening only the drift fit
(PCA-10 {R('c_s13_transition_pca10_raw_ot_whitened_drift_min', 'c_s13_transition_pca10_raw_ot_whitened_drift_max')}), whitening only the OT
cost (PCA-10 {R('c_s13_transition_pca10_whitened_ot_raw_drift_min', 'c_s13_transition_pca10_whitened_ot_raw_drift_max')}, PCA-20
{R('c_s13_transition_pca20_whitened_ot_raw_drift_min', 'c_s13_transition_pca20_whitened_ot_raw_drift_max')}).
**d**, Flow J vs the Jacobian of the generator's true expected velocity in the same PCA-15 coordinates (J_ref; S15 Step 0;
4 test datasets × J_ref neighbourhood 15/30): norm ratio {R('d_median_norm_J_over_norm_Jref_min', 'd_median_norm_J_over_norm_Jref_max')},
median per-cell cosine {rng_str(v['d_median_per_cell_cosine_min'], v['d_median_per_cell_cosine_max'], 3)}, per-cell λ₁ Spearman
{R('d_lambda1_spearman_min', 'd_lambda1_spearman_max')}, relative error after optimal scalar c* {R('d_median_relative_error_cstar_J_min', 'd_median_relative_error_cstar_J_max')}.
The reference was verified by finite differences of the true expected velocity (median per-cell cosine ≥ {f2(v['d_fd_route_cosine_min'], 3)}).
Across S14 test rows the median relative Frobenius error is {rng_str(v['d_s14_jref_relfrob_min'], v['d_s14_jref_relfrob_max'], 3)}.
**e**, Left, *post hoc*: peak-to-median ratio R of the 12-bin day-weighted strict-pseudotime λ₁ profile (reference, k = 15) on the
three transition test seeds ({R('e_R_v3_transition_min', 'e_R_v3_transition_max')}; |detected − true bin|, in parentheses in the panel, {', '.join(map(str, v['e_transition_offsets']))})
and three no-fork seeds ({R('e_R_v3_nofork_min', 'e_R_v3_nofork_max')}). Right, S14b and S14c construction (pre-registered): bootstrap stability of the true critical peak bin per reserve seed s5–s{v['e_s14c_reserve_last']}
(dashed: 0.90 rule; blue: eligible; hatched: fails only the ≥ 50-cells-per-bin rule, seeds {', '.join(map(str, v['e_only_bins_seeds']))} with smallest bins of
{', '.join(map(str, v['e_only_bins_min_bin']))} cells; ×: generation stopped by the memory guard ({v['e_n_guard_stop']} seeds); ▽: other generation failures (seeds
{', '.join(map(str, v['e_other_fail_seeds']))})). Of {v['e_s14b_n_assessed']} assessed seeds, {v['e_s14b_n_eligible']} were eligible (seeds
{', '.join(map(str, v['e_s14b_eligible_seeds']))}; none of the {v['e_s14c_n_assessed']} assessed S14c seeds) against the {v['e_required_eligible']} required, so the confirmatory transition
test could not be constructed and no model was fitted (S14b and S14c construction failures; gate values null).
""")
prov.write()